In [1]:
# notebooks/04_logistic_regression.ipynb — CELL 1

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    average_precision_score
)

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt

import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')

# -----------------------------------
# CREATE FOLDERS
# -----------------------------------
os.makedirs('../models', exist_ok=True)
os.makedirs('../plots', exist_ok=True)

# -----------------------------------
# LOAD DATASET
# -----------------------------------
df = pd.read_csv('../data/fd_loans_engineered.csv')

# -----------------------------------
# FEATURES
# -----------------------------------
FEATURES = [

    'age',
    'income_mo',
    'cibil',

    'emp_enc',
    'city_enc',
    'bank_enc',

    'fd_amount',
    'fd_tenure_mo',
    'fd_interest_rate',
    'fd_months_to_maturity',

    'loan_amt',
    'ltv',
    'loan_tenure_mo',
    'loan_interest',

    'emi',
    'emi_income',

    'num_existing_loans',
    'missed_pmts_hist',
    'pre_closure_risk',

    'relationship_yrs',
    'digital_user',

    'fd_coverage',
    'maturity_buffer',
    'interest_spread',
    'ltv_band_num',

    'high_ltv',
    'relationship_score',
    'debt_burden',
    'fd_density'
]

FEATURES = [f for f in FEATURES if f in df.columns]

TARGET = 'defaulted'

# -----------------------------------
# X AND Y
# -----------------------------------
X = df[FEATURES]
y = df[TARGET]

# -----------------------------------
# TRAIN TEST SPLIT
# -----------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -----------------------------------
# STANDARD SCALING
# -----------------------------------
scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)

X_test_sc = scaler.transform(X_test)

# -----------------------------------
# LOGISTIC REGRESSION MODEL
# -----------------------------------
lr = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=42,
    C=1.0
)

lr.fit(X_train_sc, y_train)

# -----------------------------------
# PREDICTIONS
# -----------------------------------
y_proba_lr = lr.predict_proba(X_test_sc)[:, 1]

y_pred_lr = lr.predict(X_test_sc)

# -----------------------------------
# METRICS
# -----------------------------------
auc_lr = roc_auc_score(y_test, y_proba_lr)

auc_pr_lr = average_precision_score(
    y_test,
    y_proba_lr
)

print(f"Logistic Regression AUC-ROC: {auc_lr:.4f}")

print(f"Logistic Regression AUC-PR: {auc_pr_lr:.4f}")

print("\nCLASSIFICATION REPORT:\n")

print(
    classification_report(
        y_test,
        y_pred_lr,
        target_names=['No Default', 'Default']
    )
)

# -----------------------------------
# COEFFICIENT ANALYSIS
# -----------------------------------
coef_df = pd.DataFrame({
    'Feature': FEATURES,
    'Coefficient': lr.coef_[0],
    'Odds_Ratio': np.exp(lr.coef_[0])
})

coef_df = coef_df.sort_values(
    'Coefficient',
    ascending=False
)

print("\nTOP 10 RISK FACTORS:\n")

print(
    coef_df.head(10)[
        ['Feature', 'Odds_Ratio']
    ].round(4).to_string(index=False)
)

# -----------------------------------
# SAVE MODEL
# -----------------------------------
joblib.dump(
    lr,
    '../models/lr_model.pkl'
)

joblib.dump(
    scaler,
    '../models/lr_scaler.pkl'
)

# -----------------------------------
# SAVE METRICS
# -----------------------------------
json.dump(
    {
        'auc_roc': round(auc_lr, 4),
        'auc_pr': round(auc_pr_lr, 4),
        'features': FEATURES
    },
    open('../models/lr_metrics.json', 'w'),
    indent=2
)

print("\nSaved Files:")
print("- models/lr_model.pkl")
print("- models/lr_scaler.pkl")
print("- models/lr_metrics.json")

Logistic Regression AUC-ROC: 0.6640
Logistic Regression AUC-PR: 0.1497

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

  No Default       0.94      0.64      0.76      7333
     Default       0.13      0.59      0.21       667

    accuracy                           0.64      8000
   macro avg       0.54      0.61      0.49      8000
weighted avg       0.88      0.64      0.72      8000


TOP 10 RISK FACTORS:

           Feature  Odds_Ratio
               ltv      3.1470
       fd_coverage      1.9888
         fd_amount      1.2857
  missed_pmts_hist      1.2454
  pre_closure_risk      1.1846
           emp_enc      1.1796
          high_ltv      1.1175
relationship_score      1.0609
   interest_spread      1.0425
   maturity_buffer      1.0425

Saved Files:
- models/lr_model.pkl
- models/lr_scaler.pkl
- models/lr_metrics.json
